# Qwen2.5-7B: Strictly Unified MCFIX Metrics and Pooled Regression

This notebook is one of four code-identical pipelines. Shared steps are frozen; only the model configuration and the explicitly documented large-model POS adapter differ.

## Frozen cross-model contract

- 10,011 identical rows (NR 5,007; TSR 5,004)
- Li-normalized punctuated preceding context
- `pure_token` independently tokenized and appended at the token-ID level
- context-only independent forward pass for entropy before
- previous-word controls computed before exclusions
- pooled row-wise five-fold CatBoost, split seeds 42-46

## Install pinned dependencies

In [ ]:

%pip -q install "transformers==5.13.1" "accelerate>=0.34" "catboost==1.2.10" "pycantonese==5.0.0" "openpyxl>=3.1,<4" pandas numpy scipy scikit-learn tqdm

## Configure model, inputs, and output directories

In [ ]:

from __future__ import annotations

import copy
import gc
import hashlib
import json
import math
import os
import platform
import re
import shutil
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm


# MODEL_KEY is the only intended difference among the four notebooks.
MODEL_KEY = "qwen25_7b"

MODEL_CONFIGS = {
    "jed351": {
        "label": "JED351",
        "model_id": "jed351/gpt2_tiny_zh-hk-wiki",
        "tokenizer_id": "jed351/gpt2_tiny_zh-hk-wiki",
        "metric_prefix": "jed351_",
        "pos_mode": "cifu_longest_match",
        "pos_units": "bits",
        "cifu_batch_size": 512,
    },
    "ckip_gpt2_tiny": {
        "label": "CKIP_GPT2_tiny",
        "model_id": "ckiplab/gpt2-tiny-chinese",
        "tokenizer_id": "bert-base-chinese",
        "metric_prefix": "ckip_gpt2_tiny_",
        "pos_mode": "cifu_longest_match",
        "pos_units": "bits",
        "cifu_batch_size": 512,
    },
    "qwen25_7b": {
        "label": "Qwen25_7B",
        "model_id": "Qwen/Qwen2.5-7B",
        "tokenizer_id": "Qwen/Qwen2.5-7B",
        "metric_prefix": "qwen25_7b_",
        "pos_mode": "model_vocab_pycantonese",
        "pos_units": "nats",
        "cifu_batch_size": None,
    },
    "cantonese7b": {
        "label": "CantoneseLLM7B",
        "model_id": "hon9kon9ize/CantoneseLLMChat-v1.0-7B",
        "tokenizer_id": "hon9kon9ize/CantoneseLLMChat-v1.0-7B",
        "metric_prefix": "cantonese7b_",
        "pos_mode": "legacy_column",
        "pos_units": "nats",
        "cifu_batch_size": None,
    },
}

if MODEL_KEY not in MODEL_CONFIGS:
    raise ValueError(f"Unknown MODEL_KEY: {MODEL_KEY}")
CFG = MODEL_CONFIGS[MODEL_KEY]
MODEL_ID = CFG["model_id"]
TOKENIZER_ID = CFG["tokenizer_id"]
MODEL_PREFIX = CFG["metric_prefix"]
POS_MODE = CFG["pos_mode"]

# Frozen cross-model constants.
PIPELINE_VERSION = "MCFIX_STRICT_UNIFIED_10011_V2"
MAX_SEQUENCE_TOKENS = 1024
EXPECTED_SOURCE_ROWS = 5072
EXPECTED_COMMON_ROWS = 10011
EXPECTED_TASK_COUNTS = {"NR": 5007, "TSR": 5004}
EPS = 1e-12
RUN_LIMIT_SENTENCES = None
SAVE_EVERY_SENTENCE = True

CKIP_UNSCORABLE_CANONICAL_IDS = {
    169, 322, 500, 686, 688, 799, 826, 959, 1217, 1267,
    2405, 2480, 2485, 2514, 2554, 2584, 2590, 2762, 2776, 2788,
    2824, 2874, 2877, 2878, 2896, 2904, 2983, 3049, 3089, 3190,
    3593, 3685, 3718, 3815, 3937, 4397, 4484, 4718, 4909,
}

SOURCE_FILENAME = "trad_NR_first_column_aligned_original_content.xlsx"
BASE_FILENAME = "MCFIX_NR_TSR_NEW_JED351_complete_cases.csv"
CIFU_FILENAME = "Cifu-v1_with_pycantonese_pos.csv"
required_names = [SOURCE_FILENAME, BASE_FILENAME]
if POS_MODE == "cifu_longest_match":
    required_names.append(CIFU_FILENAME)

from google.colab import drive
drive.mount("/content/drive")

INPUT_DIR = Path("/content/drive/MyDrive/MCFIX_strict_unified_inputs")
CONTENT_DIR = Path("/content")


def locate_input(filename: str) -> Path | None:
    candidates = [CONTENT_DIR / filename, INPUT_DIR / filename]
    return next((path for path in candidates if path.exists()), None)


resolved = {name: locate_input(name) for name in required_names}
missing_names = [name for name, path in resolved.items() if path is None]
if missing_names:
    from google.colab import files

    print("Upload the following files:", missing_names)
    uploaded = files.upload()
    for name in missing_names:
        if name not in uploaded:
            raise FileNotFoundError(f"Required upload was not provided: {name}")
        resolved[name] = CONTENT_DIR / name

SOURCE_PATH = resolved[SOURCE_FILENAME]
BASE_PATH = resolved[BASE_FILENAME]
CIFU_PATH = resolved.get(CIFU_FILENAME)

PIPELINE_ROOT = (
    Path("/content/drive/MyDrive")
    / f"{PIPELINE_VERSION}_{CFG['label']}"
)
EXTRACTION_DIR = PIPELINE_ROOT / "extraction"
CHECKPOINT_DIR = EXTRACTION_DIR / "sentence_checkpoints"
FINAL_DATA_DIR = PIPELINE_ROOT / "modeling_data"
REGRESSION_DIR = PIPELINE_ROOT / "regression_pooled"
for directory in (EXTRACTION_DIR, CHECKPOINT_DIR, FINAL_DATA_DIR, REGRESSION_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


print("Pipeline:", PIPELINE_VERSION)
print("Model:", MODEL_ID)
print("Tokenizer:", TOKENIZER_ID)
print("POS mode:", POS_MODE)
print("Source:", SOURCE_PATH)
print("Base:", BASE_PATH)
print("Output:", PIPELINE_ROOT)

## Build the shared source alignment and frozen 10,011-row base

In [ ]:

def read_source(path: Path) -> pd.DataFrame:
    workbook = pd.ExcelFile(path)
    sheet = "Original_Data" if "Original_Data" in workbook.sheet_names else workbook.sheet_names[0]
    frame = pd.read_excel(path, sheet_name=sheet)
    if len(frame) != EXPECTED_SOURCE_ROWS:
        raise ValueError(
            f"Expected {EXPECTED_SOURCE_ROWS:,} source rows, found {len(frame):,}."
        )
    id_column = "Unnamed: 0" if "Unnamed: 0" in frame.columns else None
    if id_column is None:
        frame["canonical_row_id"] = np.arange(len(frame), dtype=int)
    else:
        frame["canonical_row_id"] = pd.to_numeric(
            frame[id_column], errors="raise"
        ).astype(int)
    expected_ids = np.arange(EXPECTED_SOURCE_ROWS)
    if not np.array_equal(frame["canonical_row_id"].to_numpy(), expected_ids):
        raise ValueError("Source canonical_row_id must be exactly 0..5071 in row order.")
    return frame


def clean_text_value(value) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def li_text_normalizer(text) -> str:
    """Apply the released Li-style text replacements identically for all models."""
    text = clean_text_value(text)
    text = text.replace("\u22ef", "\u22ef\u22ef")
    text = text.replace("\u2026", "\u22ef\u22ef")
    text = text.replace(" ", "")
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("B612", "b612")
    text = text.replace("\u2014", "\ufe63")
    text = text.replace("\u5649", "\u5481")
    text = text.replace("\u200b", "")
    return re.compile(r"^[\u3002\u300d]+(?=[\w\u4e00-\u9fa5])").sub("", text)


def has_linguistic_char(text: str) -> bool:
    return any(unicodedata.category(char)[0] in {"L", "N"} for char in text)


POS_ALIAS = {
    "V": "VERB",
    "AD": "ADV",
    "A": "ADJ",
    "N": "NOUN",
    "R": "PRON",
    "P": "ADP",
    "C": "CCONJ",
    "U": "PART",
    "M": "NUM",
}


def canonical_pos_tag(value) -> str:
    tag = clean_text_value(value).upper()
    return POS_ALIAS.get(tag, tag)


def valid_pos_for_pos_surprisal(value) -> bool:
    tag = canonical_pos_tag(value)
    return bool(tag and tag not in {"X", "PUNCT", "SYM", "SPACE"})


def build_llm_pos(row) -> str:
    for column in ("upos*", "upos", "pos"):
        tag = canonical_pos_tag(row.get(column, ""))
        if valid_pos_for_pos_surprisal(tag):
            return tag
    return ""


SCORABLE_ALIGNMENT = {
    "exact",
    "exact_with_lexical_prefix",
    "surface_missing_target_only",
}


def split_normalized_surface(row):
    surface = li_text_normalizer(row.get("IA_LABEL", ""))
    target = li_text_normalizer(row.get("pure_token", ""))
    if not target:
        return surface, "", surface, "", "missing_pure_token"
    if not surface:
        return target, target, "", "", "surface_missing_target_only"

    starts = []
    start = surface.find(target)
    while start >= 0:
        starts.append(start)
        start = surface.find(target, start + 1)

    exact_splits = []
    lexical_prefix_splits = []
    for start in starts:
        pre_context = surface[:start]
        post_context = surface[start + len(target):]
        if not has_linguistic_char(pre_context + post_context):
            exact_splits.append((pre_context, post_context))
        elif has_linguistic_char(pre_context) and not has_linguistic_char(post_context):
            lexical_prefix_splits.append((pre_context, post_context))

    if len(exact_splits) == 1:
        pre_context, post_context = exact_splits[0]
        return surface, target, pre_context, post_context, "exact"
    if len(exact_splits) > 1:
        return surface, target, "", "", "ambiguous_repeated_target"
    if len(lexical_prefix_splits) == 1:
        pre_context, post_context = lexical_prefix_splits[0]
        return surface, target, pre_context, post_context, "exact_with_lexical_prefix"
    if len(lexical_prefix_splits) > 1:
        return surface, target, "", "", "ambiguous_repeated_target"
    return surface, target, "", "", "target_not_aligned"


source = read_source(SOURCE_PATH)
base = pd.read_csv(BASE_PATH, low_memory=False)
required_base = {
    "canonical_row_id", "TASK", "SENTENCE", "n_in_sent", "IA_LABEL",
    "pure_token", "upos*", "Freq", "N_SYL", "sentence_length_tokens",
    "LD2ROOT", "LD2HEAD", "DEPTH2ROOT", "Neighbors_Num",
    "TOTAL_DURATION", "IA_FIRST_FIXATION_DURATION",
    "IA_SECOND_FIXATION_DURATION",
}
if POS_MODE == "legacy_column":
    required_base.add("cantonese7b_pos_surprisal")
missing_base = sorted(required_base - set(base.columns))
if missing_base:
    raise KeyError(f"Base file is missing columns: {missing_base}")
if len(base) != 10089:
    raise ValueError(f"Expected 10,089 complete-case base rows, found {len(base):,}.")

# Previous-word controls are frozen from the complete original NR sequence,
# before any model-specific or complete-case exclusions.
source_order = source.sort_values(["SENTENCE", "n_in_sent"], kind="stable").copy()
source_groups = source_order.groupby("SENTENCE", sort=False, observed=True)
source_order["source_prev_freq"] = source_groups["Freq"].shift(1)
source_order["source_prev_n_syl"] = source_groups["N_SYL"].shift(1)
previous_map = source_order.set_index("canonical_row_id")[[
    "source_prev_freq", "source_prev_n_syl"
]]

common_base = base.loc[
    ~base["canonical_row_id"].isin(CKIP_UNSCORABLE_CANONICAL_IDS)
].copy().reset_index(drop=True)
if len(common_base) != EXPECTED_COMMON_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_COMMON_ROWS:,} common rows, found {len(common_base):,}."
    )
if common_base["TASK"].value_counts().to_dict() != EXPECTED_TASK_COUNTS:
    raise ValueError("The common-sample NR/TSR counts are not 5,007/5,004.")
if common_base[["canonical_row_id", "TASK"]].duplicated().any():
    raise ValueError("canonical_row_id + TASK must be unique.")

common_base = common_base.merge(
    previous_map,
    left_on="canonical_row_id",
    right_index=True,
    how="left",
    validate="many_to_one",
)
common_base["Li_Prev_Freq"] = np.where(
    common_base["source_prev_freq"].isna(),
    0.0,
    np.log(pd.to_numeric(common_base["source_prev_freq"]) + 2.0),
)
common_base["Li_Prev_N_SYL"] = pd.to_numeric(
    common_base["source_prev_n_syl"]
).fillna(0.0)

split_columns = [
    "_li_normalized_surface",
    "_llm_word_for_calc",
    "_llm_pre_context",
    "_llm_post_context",
    "_llm_alignment_status",
]
source[split_columns] = source.apply(
    lambda row: pd.Series(split_normalized_surface(row), index=split_columns),
    axis=1,
)
source["_llm_pos_for_calc"] = source.apply(build_llm_pos, axis=1)
source["_llm_metric_row_included"] = (
    source["_llm_alignment_status"].isin(SCORABLE_ALIGNMENT)
    & source["_llm_word_for_calc"].ne("")
    & source["_llm_word_for_calc"].map(has_linguistic_char)
)

unresolved = ~source["_llm_alignment_status"].isin(
    SCORABLE_ALIGNMENT | {"missing_pure_token"}
)
if unresolved.any():
    display(source.loc[unresolved, [
        "canonical_row_id", "SENTENCE", "n_in_sent", "IA_LABEL", "pure_token",
        "_llm_alignment_status",
    ]])
    raise ValueError("Unresolved IA_LABEL/pure_token alignment rows were found.")

print("Source alignment status:")
print(source["_llm_alignment_status"].value_counts(dropna=False))
print("Common sample:", len(common_base), common_base["TASK"].value_counts().to_dict())

## Load the language model and define shared token-level scoring

In [ ]:

from transformers import AutoModelForCausalLM, AutoTokenizer, BertTokenizerFast


for stale in ("model", "tokenizer", "CIFU_LEXICON", "POS_TO_TOKEN_IDS"):
    globals().pop(stale, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

INPUT_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
MODEL_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

if MODEL_KEY == "ckip_gpt2_tiny":
    tokenizer = BertTokenizerFast.from_pretrained(TOKENIZER_ID)
else:
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)
model.to(INPUT_DEVICE)
model.eval()

if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token = tokenizer.eos_token


def first_int_token_id(value):
    if isinstance(value, (list, tuple)):
        return int(value[0]) if value else None
    return int(value) if value is not None else None


start_candidates = [
    ("model.config.bos_token_id", getattr(model.config, "bos_token_id", None)),
    ("tokenizer.bos_token_id", tokenizer.bos_token_id),
    ("model.config.eos_token_id", getattr(model.config, "eos_token_id", None)),
    ("tokenizer.eos_token_id", tokenizer.eos_token_id),
    ("tokenizer.cls_token_id", tokenizer.cls_token_id),
]
START_TOKEN_SOURCE = None
START_TOKEN_ID = None
for source_name, value in start_candidates:
    candidate = first_int_token_id(value)
    if candidate is not None and 0 <= candidate < model.get_input_embeddings().num_embeddings:
        START_TOKEN_SOURCE = source_name
        START_TOKEN_ID = candidate
        break
if START_TOKEN_ID is None:
    raise RuntimeError("No valid causal start token is available for this model.")

TOKENIZER_VOCAB_BOUND = len(tokenizer)
OUTPUT_VOCAB_SIZE = int(model.get_output_embeddings().weight.shape[0])
if TOKENIZER_VOCAB_BOUND > OUTPUT_VOCAB_SIZE:
    raise RuntimeError("Tokenizer vocabulary is larger than the model output vocabulary.")


def raw_token_ids(text: str) -> list[int]:
    text = clean_text_value(text)
    if not text:
        return []
    return list(tokenizer(text, add_special_tokens=False).input_ids)


def validate_target_ids(word: str) -> list[int]:
    token_ids = raw_token_ids(word)
    if not token_ids:
        return []
    if tokenizer.unk_token_id is not None and tokenizer.unk_token_id in token_ids:
        return []
    special_ids = set(tokenizer.all_special_ids)
    if any(token_id in special_ids for token_id in token_ids):
        return []
    return token_ids


def build_prefix_ids(context_text: str, target_token_count: int = 0) -> torch.Tensor:
    context_ids = raw_token_ids(context_text)
    max_context = MAX_SEQUENCE_TOKENS - 1 - int(target_token_count)
    if max_context < 0:
        raise ValueError("Target token sequence exceeds MAX_SEQUENCE_TOKENS.")
    if len(context_ids) > max_context:
        context_ids = context_ids[-max_context:] if max_context else []
    ids = [START_TOKEN_ID, *context_ids]
    return torch.tensor([ids], dtype=torch.long, device=INPUT_DEVICE)


def mask_probability_logits(logits: torch.Tensor) -> torch.Tensor:
    logits = logits.float()
    logits = torch.nan_to_num(logits, nan=float("-inf"), posinf=float("-inf"))
    if logits.shape[-1] > TOKENIZER_VOCAB_BOUND:
        logits[..., TOKENIZER_VOCAB_BOUND:] = float("-inf")
    if not torch.isfinite(logits).any(dim=-1).all():
        raise RuntimeError("A next-token state has no finite usable logits.")
    return logits


def entropy_bits_from_logits(logits: torch.Tensor) -> float:
    logits = mask_probability_logits(logits)
    log_probs = torch.log_softmax(logits, dim=-1)
    probabilities = torch.exp(log_probs)
    contributions = torch.where(
        torch.isfinite(log_probs),
        probabilities * log_probs,
        torch.zeros_like(log_probs),
    )
    entropy = -contributions.sum(dim=-1) / math.log(2.0)
    if not torch.isfinite(entropy).all():
        raise RuntimeError("Entropy calculation produced a non-finite value.")
    return float(entropy.detach().cpu())


@torch.inference_mode()
def lexical_and_entropy_metrics(context_text: str, word: str):
    """Use the identical forced-boundary definition for every model."""
    word = clean_text_value(word)
    target_ids_list = validate_target_ids(word)
    if not target_ids_list:
        return np.nan, np.nan, np.nan, [], False, 0

    prefix_ids = build_prefix_ids(context_text, len(target_ids_list))
    prefix_mask = torch.ones_like(prefix_ids)
    prefix_logits = model(
        input_ids=prefix_ids,
        attention_mask=prefix_mask,
        use_cache=False,
    ).logits[0, -1]
    entropy_before = entropy_bits_from_logits(prefix_logits)

    target_ids = torch.tensor(
        [target_ids_list], dtype=torch.long, device=INPUT_DEVICE
    )
    full_ids = torch.cat([prefix_ids, target_ids], dim=1)
    full_mask = torch.ones_like(full_ids)
    logits = model(
        input_ids=full_ids,
        attention_mask=full_mask,
        use_cache=False,
    ).logits[0]

    prefix_length = int(prefix_ids.shape[1])
    target_count = len(target_ids_list)
    positions = torch.arange(
        prefix_length - 1,
        prefix_length - 1 + target_count,
        dtype=torch.long,
        device=logits.device,
    )
    target_logits = mask_probability_logits(logits.index_select(0, positions))
    target_log_probs = torch.log_softmax(target_logits, dim=-1)
    target_ids_flat = target_ids[0].to(logits.device)
    selected = target_log_probs[
        torch.arange(target_count, device=logits.device), target_ids_flat
    ]
    lexical_surprisal = float((-selected.sum() / math.log(2.0)).detach().cpu())
    entropy_after = entropy_bits_from_logits(logits[prefix_length + target_count - 1])

    default_full_ids = raw_token_ids(clean_text_value(context_text) + word)
    forced_raw_ids = raw_token_ids(context_text) + target_ids_list
    boundary_changed = default_full_ids != forced_raw_ids
    context_count = int(prefix_ids.shape[1] - 1)
    return (
        lexical_surprisal,
        entropy_before,
        entropy_after,
        target_ids_list,
        boundary_changed,
        context_count,
    )


print("Tokenizer class:", tokenizer.__class__.__name__)
print("Start token:", START_TOKEN_SOURCE, START_TOKEN_ID, tokenizer.decode([START_TOKEN_ID]))
print("Tokenizer vocabulary:", TOKENIZER_VOCAB_BOUND)
print("Output vocabulary:", OUTPUT_VOCAB_SIZE)
print("Model parameters:", sum(parameter.numel() for parameter in model.parameters()))

## Configure the explicitly documented POS adapter

In [ ]:

def repeat_past_key_values(past_key_values, batch_size: int):
    repeated = copy.deepcopy(past_key_values)
    if hasattr(repeated, "batch_repeat_interleave"):
        repeated.batch_repeat_interleave(batch_size)
        return repeated
    return tuple(
        tuple(state.repeat_interleave(batch_size, dim=0) for state in layer)
        for layer in repeated
    )


class CifuPOSLexicon:
    def __init__(self, path: Path, valid_pos_set):
        self.valid_pos_set = {
            canonical_pos_tag(pos)
            for pos in valid_pos_set
            if valid_pos_for_pos_surprisal(pos)
        }
        self.words = []
        self.poses = []
        self.sequences = []
        self.groups = {}
        self.ancestor_pairs_by_length = {}
        self._load(path)

    def _load(self, path: Path):
        cifu = pd.read_csv(path)
        required = {"cifu_word_clean", "pycantonese_pos", "cifu_pos_candidate_valid"}
        missing = sorted(required - set(cifu.columns))
        if missing:
            raise ValueError(f"Cifu POS file is missing columns: {missing}")
        flag = cifu["cifu_pos_candidate_valid"]
        if pd.api.types.is_bool_dtype(flag):
            valid = flag.fillna(False)
        else:
            valid = flag.fillna("").astype(str).str.lower().isin({"true", "1", "yes", "y"})
        cifu = cifu.loc[valid].copy()
        cifu["candidate_word"] = cifu["cifu_word_clean"].map(li_text_normalizer)
        cifu["candidate_pos"] = cifu["pycantonese_pos"].map(canonical_pos_tag)
        cifu = cifu[
            cifu["candidate_word"].ne("")
            & cifu["candidate_pos"].isin(self.valid_pos_set)
        ].drop_duplicates("candidate_word")

        seen = set()
        for word, pos in tqdm(
            cifu[["candidate_word", "candidate_pos"]].itertuples(index=False),
            total=len(cifu),
            desc="Tokenizing Cifu POS candidates",
        ):
            sequence = tuple(validate_target_ids(word))
            if not sequence or sequence in seen:
                continue
            seen.add(sequence)
            self.words.append(word)
            self.poses.append(pos)
            self.sequences.append(sequence)
        if not self.sequences:
            raise ValueError("No valid Cifu POS candidate survives tokenization.")

        self.poses = np.asarray(self.poses, dtype=object)
        self.lengths = np.asarray([len(value) for value in self.sequences], dtype=np.int32)
        self.max_token_length = int(self.lengths.max())
        for length in sorted(set(self.lengths.tolist())):
            indices = np.flatnonzero(self.lengths == length)
            matrix = np.asarray([self.sequences[index] for index in indices], dtype=np.int64)
            self.groups[length] = (indices, matrix)

        terminal = {sequence: index for index, sequence in enumerate(self.sequences)}
        for length, (indices, _matrix) in self.groups.items():
            descendants, ancestors = [], []
            for index in indices:
                sequence = self.sequences[index]
                for prefix_length in range(1, len(sequence)):
                    ancestor = terminal.get(sequence[:prefix_length])
                    if ancestor is not None:
                        descendants.append(index)
                        ancestors.append(ancestor)
            self.ancestor_pairs_by_length[length] = (
                np.asarray(descendants, dtype=np.int64),
                np.asarray(ancestors, dtype=np.int64),
            )

    def longest_match_exact_probabilities(self, joint_probabilities):
        exact = np.asarray(joint_probabilities, dtype=np.float64).copy()
        for length in sorted(self.groups, reverse=True):
            descendants, ancestors = self.ancestor_pairs_by_length[length]
            if len(descendants):
                np.add.at(exact, ancestors, -exact[descendants])
        materially_negative = exact < -1e-4
        if materially_negative.any():
            raise ValueError(
                "Cifu longest-match subtraction produced negative mass: "
                f"count={int(materially_negative.sum())}, minimum={float(exact.min())}"
            )
        return np.clip(exact, 0.0, None)


@torch.inference_mode()
def cifu_joint_probabilities(context_text: str):
    prefix_ids = build_prefix_ids(context_text, CIFU_LEXICON.max_token_length)
    context_output = model(input_ids=prefix_ids, use_cache=True)
    first_logits = mask_probability_logits(context_output.logits[0, -1])
    first_log_probs = torch.log_softmax(first_logits, dim=-1)
    context_past = context_output.past_key_values
    joint_log_probs = np.full(len(CIFU_LEXICON.words), -np.inf, dtype=np.float64)

    for length, (indices, matrix) in CIFU_LEXICON.groups.items():
        first_ids = torch.as_tensor(matrix[:, 0], dtype=torch.long, device=first_log_probs.device)
        first_scores = first_log_probs.index_select(0, first_ids)
        if length == 1:
            joint_log_probs[indices] = first_scores.detach().cpu().numpy()
            continue

        prefix_matrix = matrix[:, :-1]
        final_targets = matrix[:, -1]
        unique_prefixes, inverse = np.unique(prefix_matrix, axis=0, return_inverse=True)
        order = np.argsort(inverse)
        sorted_inverse = inverse[order]
        bounds = np.searchsorted(sorted_inverse, np.arange(len(unique_prefixes) + 1))

        batch_size = int(CFG["cifu_batch_size"])
        for start in range(0, len(unique_prefixes), batch_size):
            stop = min(start + batch_size, len(unique_prefixes))
            batch_prefixes = torch.as_tensor(
                unique_prefixes[start:stop], dtype=torch.long, device=INPUT_DEVICE
            )
            repeated_past = repeat_past_key_values(context_past, int(batch_prefixes.shape[0]))
            output = model(input_ids=batch_prefixes, past_key_values=repeated_past, use_cache=False)
            continuation_logits = mask_probability_logits(output.logits)
            continuation_log_probs = torch.log_softmax(continuation_logits, dim=-1)

            if batch_prefixes.shape[1] > 1:
                internal_targets = batch_prefixes[:, 1:].unsqueeze(-1)
                internal_scores = continuation_log_probs[:, :-1, :].gather(
                    2, internal_targets
                ).squeeze(-1).sum(dim=1)
            else:
                internal_scores = torch.zeros(
                    int(batch_prefixes.shape[0]),
                    dtype=continuation_log_probs.dtype,
                    device=continuation_log_probs.device,
                )

            candidate_positions = order[bounds[start]:bounds[stop]]
            local_prefix_ids = torch.as_tensor(
                inverse[candidate_positions] - start,
                dtype=torch.long,
                device=continuation_log_probs.device,
            )
            final_targets_t = torch.as_tensor(
                final_targets[candidate_positions],
                dtype=torch.long,
                device=continuation_log_probs.device,
            )
            candidate_positions_t = torch.as_tensor(
                candidate_positions, dtype=torch.long, device=first_scores.device
            )
            final_scores = continuation_log_probs[local_prefix_ids, -1, final_targets_t]
            total_scores = (
                first_scores.index_select(0, candidate_positions_t)
                + internal_scores[local_prefix_ids].to(first_scores.device)
                + final_scores.to(first_scores.device)
            )
            joint_log_probs[indices[candidate_positions]] = total_scores.detach().cpu().numpy()
    return np.exp(joint_log_probs)


def cifu_pos_surprisal(context_text: str, observed_pos: str):
    observed_pos = canonical_pos_tag(observed_pos)
    if observed_pos not in CIFU_LEXICON.valid_pos_set:
        return np.nan
    exact = CIFU_LEXICON.longest_match_exact_probabilities(
        cifu_joint_probabilities(context_text)
    )
    total_mass = float(exact.sum())
    if total_mass <= 0 or total_mass > 1.0 + 1e-4:
        raise ValueError(f"Invalid Cifu exact probability mass: {total_mass}")
    true_mass = float(exact[CIFU_LEXICON.poses == observed_pos].sum())
    if true_mass <= 0:
        return np.nan
    return -math.log2(max(true_mass / total_mass, EPS))


def remove_unicode_punctuation(text: str) -> str:
    return "".join(char for char in text if not unicodedata.category(char).startswith("P"))


def build_qwen_vocab_pos_map(valid_pos_set):
    import pycantonese

    valid_pos_set = {
        canonical_pos_tag(pos) for pos in valid_pos_set if valid_pos_for_pos_surprisal(pos)
    }
    records = []
    special_ids = set(tokenizer.all_special_ids)
    for token_id in tqdm(range(TOKENIZER_VOCAB_BOUND), desc="Tagging model vocab POS"):
        if token_id in special_ids:
            continue
        decoded = tokenizer.decode(
            [token_id], skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        text = remove_unicode_punctuation(
            clean_text_value(decoded).replace("\u2581", "").replace(" ", "")
        )
        if not text or not has_linguistic_char(text):
            continue
        try:
            tagged = pycantonese.pos_tag([text])
            raw_pos = clean_text_value(tagged[0][1]) if tagged else ""
            pos = canonical_pos_tag(POS_ALIAS.get(raw_pos.upper(), raw_pos))
        except Exception:
            continue
        if pos in valid_pos_set:
            records.append((token_id, text, pos))
    table = pd.DataFrame(records, columns=["token_id", "token_text", "token_pos"])
    if table.empty:
        raise ValueError("PyCantonese did not tag any usable model-vocab tokens.")
    mapping = {
        pos: torch.tensor(group["token_id"].tolist(), dtype=torch.long, device=INPUT_DEVICE)
        for pos, group in table.groupby("token_pos")
    }
    return mapping, table


@torch.inference_mode()
def qwen_vocab_pos_surprisal(context_text: str, observed_pos: str):
    observed_pos = canonical_pos_tag(observed_pos)
    if observed_pos not in POS_TO_TOKEN_IDS:
        return np.nan
    prefix_ids = build_prefix_ids(context_text)
    logits = mask_probability_logits(model(input_ids=prefix_ids).logits[0, -1])
    log_probs = torch.log_softmax(logits, dim=-1)
    pos_names = []
    pos_log_masses = []
    for pos, ids in POS_TO_TOKEN_IDS.items():
        if ids.numel():
            pos_names.append(pos)
            pos_log_masses.append(torch.logsumexp(log_probs.index_select(0, ids), dim=0))
    if not pos_log_masses or observed_pos not in pos_names:
        return np.nan
    stacked = torch.stack(pos_log_masses)
    total_log_mass = torch.logsumexp(stacked, dim=0)
    true_log_mass = stacked[pos_names.index(observed_pos)]
    surprisal = -(true_log_mass - total_log_mass)
    if not torch.isfinite(surprisal):
        return np.nan
    return float(surprisal.detach().cpu())


observed_pos_set = set(source.loc[source["_llm_metric_row_included"], "_llm_pos_for_calc"])
CIFU_LEXICON = None
POS_TO_TOKEN_IDS = None
MODEL_VOCAB_POS_TABLE = None
if POS_MODE == "cifu_longest_match":
    CIFU_LEXICON = CifuPOSLexicon(CIFU_PATH, observed_pos_set)
    print("Cifu POS candidates:", len(CIFU_LEXICON.words))
elif POS_MODE == "model_vocab_pycantonese":
    POS_TO_TOKEN_IDS, MODEL_VOCAB_POS_TABLE = build_qwen_vocab_pos_map(observed_pos_set)
    MODEL_VOCAB_POS_TABLE.to_csv(
        EXTRACTION_DIR / f"{MODEL_PREFIX}model_vocab_pos_map.csv",
        index=False,
        encoding="utf-8-sig",
    )
    print("Model-vocab POS candidates:", len(MODEL_VOCAB_POS_TABLE))
    pos_preflight = {
        pos: qwen_vocab_pos_surprisal("", pos)
        for pos in sorted(observed_pos_set)
    }
    nonfinite_pos = {
        pos: value for pos, value in pos_preflight.items()
        if not np.isfinite(value)
    }
    if nonfinite_pos:
        raise RuntimeError(
            "Qwen POS log-domain preflight failed: " + str(nonfinite_pos)
        )
    print("Qwen POS log-domain preflight:", pos_preflight)
elif POS_MODE != "legacy_column":
    raise ValueError(f"Unknown POS_MODE: {POS_MODE}")

## Extract exactly four analysis metrics

In [ ]:

METRIC_COLUMNS = {
    "lexical_surprisal": f"{MODEL_PREFIX}lexical_surprisal",
    "pos_surprisal": f"{MODEL_PREFIX}pos_surprisal",
    "entropy_before": f"{MODEL_PREFIX}entropy_before",
    "entropy_reduction": f"{MODEL_PREFIX}entropy_reduction",
}
AUDIT_COLUMNS = [
    f"{MODEL_PREFIX}target_token_count",
    f"{MODEL_PREFIX}target_token_ids",
    f"{MODEL_PREFIX}context_token_count",
    f"{MODEL_PREFIX}boundary_tokenization_changed",
]

signature_payload = {
    "pipeline_version": PIPELINE_VERSION,
    "model_key": MODEL_KEY,
    "model_id": MODEL_ID,
    "model_revision": getattr(model.config, "_commit_hash", None),
    "tokenizer_id": TOKENIZER_ID,
    "tokenizer_revision": tokenizer.init_kwargs.get("_commit_hash"),
    "source_sha256": sha256_file(SOURCE_PATH),
    "base_sha256": sha256_file(BASE_PATH),
    "cifu_sha256": sha256_file(CIFU_PATH) if CIFU_PATH else None,
    "target": "pure_token tokenized separately",
    "context": "Li-normalized preceding IA_LABEL text with punctuation",
    "max_sequence_tokens": MAX_SEQUENCE_TOKENS,
    "pos_mode": POS_MODE,
}
signature = hashlib.sha256(
    json.dumps(signature_payload, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
MODEL_CHECKPOINT_DIR = CHECKPOINT_DIR / signature
MODEL_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def checkpoint_path(sentence_id) -> Path:
    safe = re.sub(r"[^0-9A-Za-z_.-]+", "_", str(sentence_id))
    return MODEL_CHECKPOINT_DIR / f"sentence_{safe}.csv"


def atomic_csv(frame: pd.DataFrame, path: Path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False, encoding="utf-8-sig")
    os.replace(temporary, path)


# Preflight uses the exact production functions; entropy_before has one path only.
preflight_row = source.loc[source["_llm_metric_row_included"]].iloc[0]
preflight_values = lexical_and_entropy_metrics(
    clean_text_value(preflight_row["_llm_pre_context"]),
    clean_text_value(preflight_row["_llm_word_for_calc"]),
)
if not np.isfinite(np.asarray(preflight_values[:3], dtype=float)).all():
    raise RuntimeError(f"Metric preflight failed: {preflight_values}")
print("Metric preflight passed:", preflight_values[:3])

metric_rows = []
sentence_groups = list(source.groupby("SENTENCE", sort=False, observed=True))
if RUN_LIMIT_SENTENCES is not None:
    sentence_groups = sentence_groups[: int(RUN_LIMIT_SENTENCES)]

for sentence_id, group in tqdm(sentence_groups, desc=f"Extracting {CFG['label']} sentences"):
    group = group.sort_values("n_in_sent", kind="stable")
    path = checkpoint_path(sentence_id)
    if path.exists():
        checkpoint = pd.read_csv(path)
        expected_ids = group["canonical_row_id"].astype(int).tolist()
        if checkpoint["canonical_row_id"].astype(int).tolist() != expected_ids:
            raise RuntimeError(f"Checkpoint row mismatch for sentence {sentence_id}")
        metric_rows.extend(checkpoint.to_dict("records"))
        continue

    prefix_text = ""
    legacy_pos_prefix = ""
    sentence_rows = []
    for _, row in group.iterrows():
        surface = clean_text_value(row["_li_normalized_surface"])
        word = clean_text_value(row["_llm_word_for_calc"])
        pre_context = clean_text_value(row["_llm_pre_context"])
        pos = clean_text_value(row["_llm_pos_for_calc"])
        can_score = bool(row["_llm_metric_row_included"])
        scoring_context = prefix_text + pre_context

        lexical = pos_surprisal = entropy_before = entropy_reduction = np.nan
        target_ids = []
        boundary_changed = False
        context_token_count = 0
        if can_score:
            (
                lexical,
                entropy_before,
                entropy_after,
                target_ids,
                boundary_changed,
                context_token_count,
            ) = lexical_and_entropy_metrics(scoring_context, word)
            entropy_reduction = entropy_before - entropy_after

            if POS_MODE == "cifu_longest_match":
                pos_surprisal = cifu_pos_surprisal(scoring_context, pos)
            elif POS_MODE == "model_vocab_pycantonese":
                # This explicitly retained large-model POS method is the only
                # Qwen context exception authorized for the unified pipeline.
                pos_surprisal = qwen_vocab_pos_surprisal(legacy_pos_prefix, pos)

        record = {
            "canonical_row_id": int(row["canonical_row_id"]),
            METRIC_COLUMNS["lexical_surprisal"]: lexical,
            METRIC_COLUMNS["pos_surprisal"]: pos_surprisal,
            METRIC_COLUMNS["entropy_before"]: entropy_before,
            METRIC_COLUMNS["entropy_reduction"]: entropy_reduction,
            AUDIT_COLUMNS[0]: len(target_ids),
            AUDIT_COLUMNS[1]: " ".join(map(str, target_ids)),
            AUDIT_COLUMNS[2]: context_token_count,
            AUDIT_COLUMNS[3]: bool(boundary_changed),
        }
        sentence_rows.append(record)
        metric_rows.append(record)

        prefix_text += surface
        legacy_pos_prefix += word

    atomic_csv(pd.DataFrame(sentence_rows), path)

metrics_nr = pd.DataFrame(metric_rows)
if metrics_nr["canonical_row_id"].duplicated().any():
    raise ValueError("Extracted canonical_row_id values are not unique.")
if RUN_LIMIT_SENTENCES is None and len(metrics_nr) != EXPECTED_SOURCE_ROWS:
    raise ValueError(f"Expected {EXPECTED_SOURCE_ROWS:,} extracted source rows.")

if POS_MODE == "legacy_column":
    legacy = (
        base.loc[base["TASK"].eq("NR"), ["canonical_row_id", "cantonese7b_pos_surprisal"]]
        .drop_duplicates("canonical_row_id")
        .set_index("canonical_row_id")["cantonese7b_pos_surprisal"]
    )
    metrics_nr[METRIC_COLUMNS["pos_surprisal"]] = metrics_nr["canonical_row_id"].map(legacy)

metrics_path = EXTRACTION_DIR / f"{CFG['label']}_NR_strict_unified_metrics.csv"
atomic_csv(metrics_nr, metrics_path)
print("Saved NR metrics:", metrics_path)
print("Boundary-changed rows:", int(metrics_nr[AUDIT_COLUMNS[3]].sum()))

## Map NR text metrics to both tasks and validate the common sample

In [ ]:

metric_map = metrics_nr.set_index("canonical_row_id")
final_data = common_base.copy()

# The supplied base contains legacy model columns. They are input material, not
# outputs of this frozen pipeline, so remove them before adding the current
# model's newly extracted four metrics.
legacy_metric_prefixes = (
    "jed351_", "ckip_gpt2_tiny_", "qwen25_7b_", "cantonese7b_"
)
legacy_metric_columns = [
    column for column in final_data.columns
    if column.startswith(legacy_metric_prefixes)
]
if legacy_metric_columns:
    final_data = final_data.drop(columns=legacy_metric_columns)

drop_existing = [
    column for column in [*METRIC_COLUMNS.values(), *AUDIT_COLUMNS]
    if column in final_data.columns
]
if drop_existing:
    final_data = final_data.drop(columns=drop_existing)

for column in [*METRIC_COLUMNS.values(), *AUDIT_COLUMNS]:
    final_data[column] = final_data["canonical_row_id"].map(metric_map[column])

primary_numeric = final_data[list(METRIC_COLUMNS.values())].apply(
    pd.to_numeric, errors="coerce"
)
missing_counts = primary_numeric.isna().sum()
if missing_counts.any():
    raise ValueError(
        "Model metrics are incomplete on the frozen 10,011-row sample:\n"
        + missing_counts[missing_counts.gt(0)].to_string()
    )
if not np.isfinite(primary_numeric.to_numpy(dtype=float)).all():
    raise ValueError("Non-finite model metrics remain on the common sample.")
final_data[list(METRIC_COLUMNS.values())] = primary_numeric

# Cross-task copies must be identical because text-derived metrics are extracted once from NR.
for column in METRIC_COLUMNS.values():
    wide = final_data.pivot(index="canonical_row_id", columns="TASK", values=column)
    paired = wide.dropna(subset=["NR", "TSR"])
    if not np.allclose(paired["NR"], paired["TSR"], rtol=0.0, atol=0.0):
        raise RuntimeError(f"NR/TSR mapped values differ for {column}")

expected_keys = common_base[["canonical_row_id", "TASK"]].reset_index(drop=True)
actual_keys = final_data[["canonical_row_id", "TASK"]].reset_index(drop=True)
if not actual_keys.equals(expected_keys):
    raise RuntimeError("Final row identity/order changed during metric mapping.")

FINAL_MODELING_PATH = (
    FINAL_DATA_DIR / f"MCFIX_NR_TSR_{CFG['label']}_strict_unified_10011.csv"
)
FINAL_XLSX_PATH = FINAL_MODELING_PATH.with_suffix(".xlsx")
final_data.to_csv(FINAL_MODELING_PATH, index=False, encoding="utf-8-sig")
final_data.to_excel(FINAL_XLSX_PATH, index=False)

extraction_manifest = {
    **signature_payload,
    "signature": signature,
    "common_rows": len(final_data),
    "task_counts": final_data["TASK"].value_counts().to_dict(),
    "metric_columns": METRIC_COLUMNS,
    "metric_units": {
        "lexical_surprisal": "bits",
        "pos_surprisal": CFG["pos_units"],
        "entropy_before": "bits",
        "entropy_reduction": "bits",
    },
    "previous_controls": "computed on the complete 5,072-row source before exclusions",
    "token_boundary": "context and pure_token tokenized separately; IDs concatenated",
    "entropy_before": "independent context-only forward pass",
    "boundary_changed_source_rows": int(metrics_nr[AUDIT_COLUMNS[3]].sum()),
    "final_csv": str(FINAL_MODELING_PATH),
}
(PIPELINE_ROOT / "extraction_manifest.json").write_text(
    json.dumps(extraction_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Final modeling CSV:", FINAL_MODELING_PATH)
print("Final modeling XLSX:", FINAL_XLSX_PATH)

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Pooled NR+TSR regression and complete ablations

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import platform
import shutil
import time
from pathlib import Path

import catboost
import numpy as np
import pandas as pd
import scipy
import sklearn
from catboost import CatBoostRegressor
from scipy.stats import pearsonr, spearmanr, t as student_t
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from tqdm.auto import tqdm


# Prespecified robustness design.
SPLIT_SEEDS = (42, 43, 44, 45, 46)
N_SPLITS = 5
CATBOOST_RANDOM_SEED = 10
CATBOOST_ITERATIONS = 1000
TASK_TYPE = "GPU"                 # Use "CPU" for exact CPU-mode replication.
GPU_DEVICES = "0"
CPU_THREAD_COUNT = 4
SAVE_EVERY_N_JOBS = 5

# Persistent output is strongly recommended for this long run.
USE_GOOGLE_DRIVE = False
DRIVE_OUTPUT_DIR = str(REGRESSION_DIR)
LOCAL_OUTPUT_DIR = str(REGRESSION_DIR)

# Leave blank to upload the supplied complete-case CSV when prompted.
INPUT_PATH = str(FINAL_MODELING_PATH)

TASKS = ("NR", "TSR")
TARGETS = {
    "FFD": "IA_FIRST_FIXATION_DURATION",
    "SFD": "IA_SECOND_FIXATION_DURATION",
    "TFD": "TOTAL_DURATION",
}
LLM_METRICS = (
    "lexical_surprisal",
    "pos_surprisal",
    "entropy_before",
    "entropy_reduction",
)
METRIC_COLUMNS = {
    metric: f"{MODEL_PREFIX}{metric}" for metric in LLM_METRICS
}

print("Configuration loaded.")
print("Partition seeds:", SPLIT_SEEDS)
print("CatBoost task type:", TASK_TYPE)

In [ ]:
OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
input_path = Path(INPUT_PATH)
if not input_path.exists():
    raise FileNotFoundError(input_path)
print("Regression input:", input_path)
print("Regression output:", OUTPUT_DIR)

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def regression_metrics(observed, predicted) -> dict[str, float]:
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mse = float(mean_squared_error(observed, predicted))
    if np.std(observed) > 1e-12 and np.std(predicted) > 1e-12:
        pearson = float(pearsonr(observed, predicted).statistic)
        spearman = float(spearmanr(observed, predicted).statistic)
    else:
        pearson = math.nan
        spearman = math.nan
    return {
        "mae": float(mean_absolute_error(observed, predicted)),
        "mse": mse,
        "rmse": float(math.sqrt(mse)),
        "r2": float(r2_score(observed, predicted)),
        "pearson": pearson,
        "spearman": spearman,
    }


def prepare_data(path: Path):
    data = pd.read_csv(path, low_memory=False)
    required = {
        "canonical_row_id", "TASK", "SENTENCE", "n_in_sent",
        "sentence_length_tokens", "pure_token", "upos*", "LD2ROOT",
        "LD2HEAD", "DEPTH2ROOT", "Neighbors_Num", "N_SYL", "Freq",
        "Li_Prev_Freq", "Li_Prev_N_SYL",
        *METRIC_COLUMNS.values(), *TARGETS.values(),
    }
    missing = sorted(required.difference(data.columns))
    if missing:
        raise KeyError(f"Required columns are missing: {missing}")

    data["TASK"] = data["TASK"].astype(str)
    if set(data["TASK"].dropna()) != set(TASKS):
        raise ValueError("TASK must contain exactly NR and TSR.")
    if data[["canonical_row_id", "TASK"]].duplicated().any():
        raise ValueError("canonical_row_id must be unique within each task.")

    data["TASK_indicator"] = data["TASK"].eq("TSR").astype(np.int8)
    data["Li_log_Freq"] = np.log(pd.to_numeric(data["Freq"]) + 1.0)
    data["Li_N_SYL"] = pd.to_numeric(data["N_SYL"])
    data["Li_Word_Position"] = (
        (pd.to_numeric(data["n_in_sent"]) + 1.0)
        / pd.to_numeric(data["sentence_length_tokens"])
    )

    # Frozen upstream controls; every model reads these identical values.
    data["Li_Prev_Freq"] = pd.to_numeric(data["Li_Prev_Freq"], errors="raise")
    data["Li_Prev_N_SYL"] = pd.to_numeric(data["Li_Prev_N_SYL"], errors="raise")

    data["upos_model"] = (
        data["upos*"].fillna("MISSING").astype(str).replace({"V": "VERB"})
    )
    pos_dummies = pd.get_dummies(
        data["upos_model"], prefix="CPOS", dtype=np.int8
    )
    data = pd.concat([data, pos_dummies], axis=1)

    blocks = {
        "TASK": ["TASK_indicator"],
        "POS": pos_dummies.columns.tolist(),
        "LD2ROOT": ["LD2ROOT"],
        "LD2HEAD": ["LD2HEAD"],
        "DEPTH2ROOT": ["DEPTH2ROOT"],
        "Neighbors_Num": ["Neighbors_Num"],
        "Frequency": ["Li_log_Freq"],
        "Syllable_Number": ["Li_N_SYL"],
        "Previous_Frequency": ["Li_Prev_Freq"],
        "Previous_Syllable_Number": ["Li_Prev_N_SYL"],
        "Word_Position": ["Li_Word_Position"],
    }

    numeric_columns = (
        [column for columns in blocks.values() for column in columns]
        + list(METRIC_COLUMNS.values())
        + list(TARGETS.values())
    )
    numeric = data[numeric_columns].apply(pd.to_numeric, errors="coerce")
    if numeric.isna().any().any():
        counts = numeric.isna().sum()
        raise ValueError(
            "Missing modeling values:\n" + counts[counts.gt(0)].to_string()
        )
    if not np.isfinite(numeric.to_numpy(dtype=float)).all():
        raise ValueError("Non-finite modeling values were found.")
    data[numeric_columns] = numeric
    return data.reset_index(drop=True), blocks


def add_fold_assignments(data: pd.DataFrame) -> pd.DataFrame:
    """Frozen CKIP-style row-wise pooled KFold for all four models."""
    data = data.copy()
    for seed in SPLIT_SEEDS:
        folds = np.full(len(data), -1, dtype=np.int8)
        splitter = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        for fold, (_train, test) in enumerate(splitter.split(data), start=1):
            folds[test] = fold
        data[f"pooled_fold_seed_{seed}"] = folds
    fold_columns = [f"pooled_fold_seed_{seed}" for seed in SPLIT_SEEDS]
    if data[fold_columns].lt(1).any().any():
        raise RuntimeError("Incomplete pooled fold assignment.")
    return data

data, FEATURE_BLOCKS = prepare_data(input_path)
data = add_fold_assignments(data)

if len(data) != EXPECTED_COMMON_ROWS:
    raise ValueError(f"Expected {EXPECTED_COMMON_ROWS:,} common rows, found {len(data):,}.")
expected_task_counts = EXPECTED_TASK_COUNTS
actual_task_counts = data["TASK"].value_counts().to_dict()
if actual_task_counts != expected_task_counts:
    raise ValueError(
        f"Unexpected task counts: {actual_task_counts}; "
        f"expected {expected_task_counts}."
    )

fold_export_columns = [
    "canonical_row_id", "TASK", "SENTENCE", "n_in_sent", "pure_token",
    *[
        column for column in data.columns
        if column.startswith("pooled_fold_seed_")
    ],
]
data[fold_export_columns].to_csv(
    TABLES_DIR / "fold_assignments.csv", index=False, encoding="utf-8-sig"
)

print(f"Prepared {len(data):,} rows:", actual_task_counts)
print("POS dummy columns:", len(FEATURE_BLOCKS["POS"]))

In [ ]:
def baseline_columns(blocks, pooled: bool) -> list[str]:
    block_names = list(blocks)
    if not pooled:
        block_names.remove("TASK")
    return [column for name in block_names for column in blocks[name]]


def build_specifications(blocks, pooled: bool):
    baseline = baseline_columns(blocks, pooled)
    all_metric_columns = list(METRIC_COLUMNS.values())
    specifications = {"baseline_no_llm": baseline}
    rows = [{
        "specification": "baseline_no_llm",
        "analysis_type": "baseline",
        "changed_block": "NONE",
        "n_features": len(baseline),
    }]

    specifications["all_four"] = baseline + all_metric_columns
    rows.append({
        "specification": "all_four",
        "analysis_type": "full_all_four",
        "changed_block": "NONE",
        "n_features": len(specifications["all_four"]),
    })

    for metric, column in METRIC_COLUMNS.items():
        name = f"add_{metric}"
        specifications[name] = baseline + [column]
        rows.append({
            "specification": name,
            "analysis_type": "single_metric_addition",
            "changed_block": metric,
            "n_features": len(specifications[name]),
        })

    for metric, column in METRIC_COLUMNS.items():
        name = f"all_four_minus_{metric}"
        specifications[name] = baseline + [
            value for value in all_metric_columns if value != column
        ]
        rows.append({
            "specification": name,
            "analysis_type": "leave_one_metric_out",
            "changed_block": metric,
            "n_features": len(specifications[name]),
        })

    for block_name, removed_columns in blocks.items():
        if not pooled and block_name == "TASK":
            continue
        removed = set(removed_columns)
        name = f"all_four_minus_{block_name}"
        specifications[name] = [
            column for column in baseline if column not in removed
        ] + all_metric_columns
        rows.append({
            "specification": name,
            "analysis_type": "leave_one_control_block_out",
            "changed_block": block_name,
            "n_features": len(specifications[name]),
        })
    return specifications, pd.DataFrame(rows)


DESIGNS = (
    ("pooled_with_TASK", "POOLED", True),
)

SPECIFICATIONS = {}
specification_frames = []
for design, training_task, pooled in DESIGNS:
    specs, frame = build_specifications(FEATURE_BLOCKS, pooled)
    SPECIFICATIONS[(design, training_task)] = specs
    frame.insert(0, "training_task", training_task)
    frame.insert(0, "design", design)
    specification_frames.append(frame)

feature_specifications = pd.concat(specification_frames, ignore_index=True)
feature_specifications.to_csv(
    TABLES_DIR / "feature_specifications.csv",
    index=False,
    encoding="utf-8-sig",
)

jobs = []
for split_seed in SPLIT_SEEDS:
    for design, training_task, _ in DESIGNS:
        for specification, columns in SPECIFICATIONS[(design, training_task)].items():
            metadata = feature_specifications.loc[
                feature_specifications["design"].eq(design)
                & feature_specifications["training_task"].eq(training_task)
                & feature_specifications["specification"].eq(specification)
            ].iloc[0]
            for target, target_column in TARGETS.items():
                jobs.append({
                    "split_seed": split_seed,
                    "design": design,
                    "training_task": training_task,
                    "specification": specification,
                    "analysis_type": metadata["analysis_type"],
                    "changed_block": metadata["changed_block"],
                    "feature_columns": columns,
                    "target": target,
                    "target_column": target_column,
                })

print(f"Model jobs: {len(jobs):,}")
print(f"CatBoost fits: {len(jobs) * N_SPLITS:,}")
if len(jobs) != 315:
    raise RuntimeError(f"Expected 315 pooled jobs, found {len(jobs)}.")
display(feature_specifications)

In [ ]:
COMPLETED_PATH = TABLES_DIR / "completed_jobs.csv"
OOF_PATH = TABLES_DIR / "all_oof_metrics.csv"
FOLD_PATH = TABLES_DIR / "all_fold_metrics.csv"
CONFIG_PATH = OUTPUT_DIR / "run_config.json"

input_sha256 = sha256_file(input_path)
run_config = {
    "input_file": input_path.name,
    "input_sha256": input_sha256,
    "n_rows": int(len(data)),
    "task_counts": actual_task_counts,
    "split_seeds": list(SPLIT_SEEDS),
    "n_splits": N_SPLITS,
    "catboost_random_seed": CATBOOST_RANDOM_SEED,
    "catboost_iterations": CATBOOST_ITERATIONS,
    "task_type": TASK_TYPE,
    "gpu_devices": GPU_DEVICES if TASK_TYPE.upper() == "GPU" else None,
    "cpu_thread_count": CPU_THREAD_COUNT,
    "catboost_version": catboost.__version__,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "scipy_version": scipy.__version__,
    "sklearn_version": sklearn.__version__,
    "python_version": platform.python_version(),
    "delta_mae_definition": "baseline_mae - augmented_model_mae",
    "ablation_definition": "removed_model_mae - full_all_four_mae",
    "metric_units": extraction_manifest["metric_units"],
    "pipeline_version": PIPELINE_VERSION,
    "model_key": MODEL_KEY,
    "row_identity_sha256": hashlib.sha256(
        data[["canonical_row_id", "TASK"]].to_csv(index=False).encode("utf-8")
    ).hexdigest(),
}

if CONFIG_PATH.exists():
    existing_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    protected_keys = (
        "input_sha256", "split_seeds", "n_splits", "catboost_random_seed",
        "catboost_iterations", "task_type", "gpu_devices",
    )
    mismatches = {
        key: (existing_config.get(key), run_config.get(key))
        for key in protected_keys
        if existing_config.get(key) != run_config.get(key)
    }
    if mismatches:
        raise RuntimeError(
            "The existing output directory contains a different run "
            f"configuration: {mismatches}"
        )
else:
    CONFIG_PATH.write_text(
        json.dumps(run_config, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

completed = (
    pd.read_csv(COMPLETED_PATH) if COMPLETED_PATH.exists() else pd.DataFrame()
)
oof_rows = pd.read_csv(OOF_PATH).to_dict("records") if OOF_PATH.exists() else []
fold_rows = (
    pd.read_csv(FOLD_PATH).to_dict("records") if FOLD_PATH.exists() else []
)

job_key_columns = [
    "split_seed", "design", "training_task", "specification", "target"
]
if completed.empty:
    completed_keys = set()
else:
    completed_keys = set(
        map(tuple, completed[job_key_columns].itertuples(index=False, name=None))
    )


def persist_progress(completed_records):
    def atomic_csv(frame, path):
        temporary = path.with_suffix(".tmp.csv")
        frame.to_csv(temporary, index=False, encoding="utf-8-sig")
        temporary.replace(path)

    completed_frame = pd.DataFrame(completed_records).drop_duplicates(
        subset=job_key_columns, keep="last"
    )
    oof_frame = pd.DataFrame(oof_rows).drop_duplicates(
        subset=job_key_columns + ["evaluation_scope"], keep="last"
    )
    fold_frame = pd.DataFrame(fold_rows).drop_duplicates(
        subset=job_key_columns + ["evaluation_scope", "fold"], keep="last"
    )

    # Write metric tables first and the completion ledger last. This makes
    # interrupted Drive writes safe to resume without skipping an incomplete job.
    atomic_csv(oof_frame, OOF_PATH)
    atomic_csv(fold_frame, FOLD_PATH)
    atomic_csv(completed_frame, COMPLETED_PATH)


def analysis_subset(job):
    if job["training_task"] != "POOLED":
        raise ValueError("This frozen pipeline supports pooled training only.")
    subset = data.reset_index(drop=True)
    fold_column = f"pooled_fold_seed_{job['split_seed']}"
    evaluation_scopes = {
        "POOLED": np.ones(len(subset), dtype=bool),
        "NR_within_pooled": subset["TASK"].eq("NR").to_numpy(),
        "TSR_within_pooled": subset["TASK"].eq("TSR").to_numpy(),
    }
    return subset, fold_column, evaluation_scopes


def make_catboost_model():
    kwargs = {
        "iterations": CATBOOST_ITERATIONS,
        "random_seed": CATBOOST_RANDOM_SEED,
        "verbose": False,
        "allow_writing_files": False,
    }
    if TASK_TYPE.upper() == "GPU":
        kwargs.update({"task_type": "GPU", "devices": GPU_DEVICES})
    else:
        kwargs.update({"task_type": "CPU", "thread_count": CPU_THREAD_COUNT})
    return CatBoostRegressor(**kwargs)


completed_records = completed.to_dict("records") if not completed.empty else []
pending_jobs = [
    job for job in jobs
    if tuple(job[column] for column in job_key_columns) not in completed_keys
]

print(f"Already completed: {len(completed_keys):,}/{len(jobs):,}")
print(f"Pending: {len(pending_jobs):,}")

started_all = time.perf_counter()
since_save = 0
progress = tqdm(pending_jobs, unit="model job")
for job in progress:
    subset, fold_column, evaluation_scopes = analysis_subset(job)
    predictors = subset[job["feature_columns"]].to_numpy(dtype=float)
    observed = subset[job["target_column"]].to_numpy(dtype=np.float64)
    folds = subset[fold_column].to_numpy(dtype=np.int8)
    predicted = np.full(len(subset), np.nan, dtype=np.float64)
    job_started = time.perf_counter()

    for fold in range(1, N_SPLITS + 1):
        test = np.flatnonzero(folds == fold)
        train = np.flatnonzero(folds != fold)
        model = make_catboost_model()
        model.fit(predictors[train], observed[train])
        predicted[test] = np.asarray(model.predict(predictors[test]), dtype=float)
        del model
        gc.collect()

    if np.isnan(predicted).any():
        raise RuntimeError(f"Incomplete OOF predictions for {job}")

    common = {
        "split_seed": job["split_seed"],
        "design": job["design"],
        "training_task": job["training_task"],
        "specification": job["specification"],
        "analysis_type": job["analysis_type"],
        "changed_block": job["changed_block"],
        "target": job["target"],
        "n_features": len(job["feature_columns"]),
    }
    for evaluation_scope, scope_mask in evaluation_scopes.items():
        oof_rows.append({
            **common,
            "evaluation_scope": evaluation_scope,
            "n": int(scope_mask.sum()),
            **regression_metrics(observed[scope_mask], predicted[scope_mask]),
        })
        for fold in range(1, N_SPLITS + 1):
            mask = scope_mask & (folds == fold)
            fold_rows.append({
                **common,
                "evaluation_scope": evaluation_scope,
                "fold": fold,
                "n": int(mask.sum()),
                **regression_metrics(observed[mask], predicted[mask]),
            })

    completed_records.append({
        **{column: job[column] for column in job_key_columns},
        "seconds": time.perf_counter() - job_started,
    })
    completed_keys.add(tuple(job[column] for column in job_key_columns))
    since_save += 1
    progress.set_postfix(
        seed=job["split_seed"],
        scope=job["training_task"],
        target=job["target"],
    )

    if since_save >= SAVE_EVERY_N_JOBS:
        persist_progress(completed_records)
        since_save = 0

persist_progress(completed_records)
run_config["completed_jobs"] = len(completed_keys)
run_config["expected_jobs"] = len(jobs)
run_config["latest_session_seconds"] = time.perf_counter() - started_all
CONFIG_PATH.write_text(
    json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"Completed jobs: {len(completed_keys):,}/{len(jobs):,}")
print(f"Latest session: {(time.perf_counter() - started_all) / 60:.1f} minutes")

In [ ]:
oof = pd.read_csv(OOF_PATH)
if len(completed_keys) != len(jobs):
    raise RuntimeError(
        f"Only {len(completed_keys):,}/{len(jobs):,} jobs are complete. "
        "Finish the previous cell before summarizing."
    )

merge_keys = [
    "split_seed", "design", "training_task", "evaluation_scope", "target"
]
baseline = (
    oof.loc[oof["specification"].eq("baseline_no_llm"), merge_keys + ["mae"]]
    .rename(columns={"mae": "baseline_mae"})
)

addition = oof.loc[
    oof["analysis_type"].isin(["single_metric_addition", "full_all_four"])
].merge(baseline, on=merge_keys, validate="many_to_one")
addition["delta_mae"] = addition["baseline_mae"] - addition["mae"]
addition.to_csv(
    TABLES_DIR / "metric_additions_by_seed.csv",
    index=False,
    encoding="utf-8-sig",
)

full = (
    oof.loc[oof["specification"].eq("all_four"), merge_keys + ["mae"]]
    .rename(columns={"mae": "full_all_four_mae"})
)
ablation = oof.loc[
    oof["analysis_type"].isin(
        ["leave_one_metric_out", "leave_one_control_block_out"]
    )
].merge(full, on=merge_keys, validate="many_to_one")
ablation["mae_increase_when_removed"] = (
    ablation["mae"] - ablation["full_all_four_mae"]
)
ablation.to_csv(
    TABLES_DIR / "leave_one_out_ablation_by_seed.csv",
    index=False,
    encoding="utf-8-sig",
)


def robustness_summary(frame, value_column, group_columns):
    summary = (
        frame.groupby(group_columns, dropna=False, observed=True)[value_column]
        .agg(["count", "mean", "std", "median", "min", "max"])
        .reset_index()
    )
    summary = summary.rename(columns={
        "count": "n_split_seeds",
        "mean": f"mean_{value_column}",
        "std": f"sd_{value_column}",
        "median": f"median_{value_column}",
        "min": f"min_{value_column}",
        "max": f"max_{value_column}",
    })
    positive = (
        frame.assign(_positive=frame[value_column].gt(0).astype(int))
        .groupby(group_columns, dropna=False, observed=True)["_positive"]
        .sum()
        .reset_index(name="positive_seed_count")
    )
    summary = summary.merge(positive, on=group_columns, validate="one_to_one")
    n = summary["n_split_seeds"].astype(float)
    sem = summary[f"sd_{value_column}"] / np.sqrt(n)
    critical = student_t.ppf(0.975, np.maximum(n - 1.0, 1.0))
    margin = critical * sem
    summary["ci95_low"] = summary[f"mean_{value_column}"] - margin
    summary["ci95_high"] = summary[f"mean_{value_column}"] + margin
    summary["positive_seed_fraction"] = (
        summary["positive_seed_count"].astype(str)
        + "/"
        + summary["n_split_seeds"].astype(str)
    )
    return summary


addition_groups = [
    "design", "training_task", "evaluation_scope", "target",
    "specification", "analysis_type", "changed_block",
]
addition_summary = robustness_summary(
    addition, "delta_mae", addition_groups
)
addition_summary.to_csv(
    TABLES_DIR / "metric_additions_robustness.csv",
    index=False,
    encoding="utf-8-sig",
)

ablation_groups = [
    "design", "training_task", "evaluation_scope", "target",
    "specification", "analysis_type", "changed_block",
]
ablation_summary = robustness_summary(
    ablation, "mae_increase_when_removed", ablation_groups
)
ablation_summary.to_csv(
    TABLES_DIR / "leave_one_out_ablation_robustness.csv",
    index=False,
    encoding="utf-8-sig",
)

performance_groups = [
    "design", "training_task", "evaluation_scope", "target",
    "specification", "analysis_type", "changed_block",
]
performance_summary = (
    oof.groupby(performance_groups, dropna=False, observed=True)
    [["mae", "mse", "rmse", "r2", "pearson", "spearman"]]
    .agg(["mean", "std", "min", "max"])
)
performance_summary.columns = [
    f"{metric}_{stat}" for metric, stat in performance_summary.columns
]
performance_summary.reset_index().to_csv(
    TABLES_DIR / "all_performance_metrics_robustness.csv",
    index=False,
    encoding="utf-8-sig",
)

main_scopes = addition_summary.loc[
    addition_summary["evaluation_scope"].isin(["POOLED", "NR", "TSR"])
].copy()
display_columns = [
    "training_task", "target", "specification", "changed_block",
    "mean_delta_mae", "sd_delta_mae", "min_delta_mae", "max_delta_mae",
    "ci95_low", "ci95_high", "positive_seed_fraction",
]
display(
    main_scopes[display_columns]
    .sort_values(["training_task", "target", "changed_block"])
    .reset_index(drop=True)
)

## Package and download results

In [ ]:

release_dir = Path("/content") / f"{PIPELINE_VERSION}_{CFG['label']}_RESULTS"
if release_dir.exists():
    shutil.rmtree(release_dir)
release_dir.mkdir(parents=True)

for source_path in (FINAL_MODELING_PATH, FINAL_XLSX_PATH, PIPELINE_ROOT / "extraction_manifest.json"):
    shutil.copy2(source_path, release_dir / source_path.name)
shutil.copytree(REGRESSION_DIR / "tables", release_dir / "regression_tables")
shutil.copy2(REGRESSION_DIR / "run_config.json", release_dir / "run_config.json")

archive = Path(shutil.make_archive(str(release_dir), "zip", root_dir=release_dir))
print("Result ZIP:", archive)
print(f"ZIP size: {archive.stat().st_size / (1024 ** 2):.2f} MB")

from google.colab import files
files.download(str(archive))